In [1]:
import random
import pandas as pd
import sys
sys.path.append('../')

from src.cipher_utils.classical import (
    caesar_encrypt, vigenere_encrypt,
    rail_fence_encrypt, columnar_transposition_encrypt
)


In [2]:
word_bank = ["HELLO", "WORLD", "CRYPTO", "ATTACK", "DEFENSE", "MESSAGE", "PYTHON", "SECURE", "HACKER", "SECRET"]

def generate_plaintext():
    return ' '.join(random.choices(word_bank, k=random.randint(3, 6)))


In [3]:
def encrypt_sample(text, method):
    if method == "caesar":
        key = random.randint(1, 25)
        cipher = caesar_encrypt(text, key)
    elif method == "vigenere":
        key = ''.join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=random.randint(3, 6)))
        cipher = vigenere_encrypt(text, key)
    elif method == "rail_fence":
        key = random.randint(2, 5)
        cipher = rail_fence_encrypt(text, key)
    elif method == "columnar":
        key = ''.join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=random.randint(3, 6)))
        cipher = columnar_transposition_encrypt(text, key)
    else:
        raise ValueError("Unknown method")
    return cipher, key


In [4]:
methods = ["caesar", "vigenere", "rail_fence", "columnar"]
data = []

for _ in range(10000):  # You can adjust this number
    method = random.choice(methods)
    plain = generate_plaintext()
    cipher, key = encrypt_sample(plain, method)
    data.append({
        "plain_text": plain,
        "cipher_text": cipher,
        "method": method,
        "key": key
    })

df = pd.DataFrame(data)
df.head()


,plain_text,cipher_text,method,key
0,CRYPTO SECRET MESSAGE MESSAGE,CYT ERTMSAEMSAERPOSCE ESG ESG,rail_fence,2
1,PYTHON WORLD ATTACK,YWAKHRTTOTOLAPNDC,columnar,RCHEH
2,HELLO ATTACK SECRET HELLO HACKER,GDKKN ZSSZBJ RDBQDS GDKKN GZBJDQ,caesar,25
3,HELLO HACKER MESSAGE,OESLCEEEAMGHHRALKS,columnar,MIGXB
4,HACKER DEFENSE DEFENSE,KDFNHU GHIHQVH GHIHQVH,caesar,3


In [ ]:
import os
# Make sure the data directory exists
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/train.csv", index=False)
print("Saved dataset to data/train.csv")


Saved dataset to data/train.csv


In [8]:
def columnar_encrypt(plaintext, key):
    # Remove spaces and lowercase
    plaintext = ''.join(plaintext.lower().split())
    n_cols = len(key)
    n_rows = (len(plaintext) + n_cols - 1) // n_cols  # ceil division
    grid = ['' for _ in range(n_cols)]

    for i, char in enumerate(plaintext):
        col = i % n_cols
        grid[col] += char

    # Determine column order based on sorted key
    key_order = sorted(range(len(key)), key=lambda x: key[x])
    ciphertext = ''.join(grid[i] for i in key_order)
    return ciphertext


In [9]:
import random
import string
import pandas as pd
import os
from collections import Counter

#  Helpers 
def distort_text(text, distortion_rate=0.1):
    text = list(text)
    num_changes = max(1, int(len(text) * distortion_rate))
    for _ in range(num_changes):
        i = random.randint(0, len(text) - 1)
        op = random.choice(['flip', 'swap', 'delete', 'insert'])
        if op == 'flip':
            text[i] = random.choice(string.ascii_lowercase)
        elif op == 'swap' and i < len(text) - 1:
            text[i], text[i+1] = text[i+1], text[i]
        elif op == 'delete':
            del text[i]
        elif op == 'insert':
            text.insert(i, random.choice(string.ascii_lowercase))
    return ''.join(text)

def get_text_metadata(text):
    freqs = Counter(text)
    total_chars = len(text)
    avg_freq = sum(freqs.values()) / len(freqs)
    return {
        "ciphertext_length": total_chars,
        "avg_char_freq": avg_freq,
        "unique_chars": len(freqs)
    }

# Key Generators
def random_vigenere_key():
    return ''.join(random.choices(string.ascii_lowercase, k=random.randint(3, 10)))

def random_columnar_key():
    return ''.join(random.sample(string.ascii_lowercase, k=random.randint(4, 8)))

# There already exist cipher functions from earlier which are as follows:
# caesar_encrypt(), vigenere_encrypt(), rail_fence_encrypt(), columnar_encrypt()

# Enhanced Dataset Generator 
def generate_dataset_enhanced(num_samples=1000):
    plaintexts = ["this is a secret message number " + str(i) for i in range(num_samples)]
    data = []
    for pt in plaintexts:
        cipher_type = random.choice(['caesar', 'vigenere', 'rail_fence', 'columnar'])

        if cipher_type == 'caesar':
            key = random.randint(1, 25)
            ct = caesar_encrypt(pt, key)
        elif cipher_type == 'vigenere':
            key = random_vigenere_key()
            ct = vigenere_encrypt(pt, key)
        elif cipher_type == 'rail_fence':
            key = random.randint(2, 5)
            ct = rail_fence_encrypt(pt, key)
        elif cipher_type == 'columnar':
            key = random_columnar_key()
            ct = columnar_encrypt(pt, key)
        
        # Add distortion
        noisy_ct = distort_text(ct, distortion_rate=0.1)

        # Metadata
        meta = get_text_metadata(noisy_ct)

        data.append({
            "plaintext": pt,
            "ciphertext": noisy_ct,
            "cipher_type": cipher_type,
            "key": key,
            **meta
        })

    df = pd.DataFrame(data)
    os.makedirs("../data", exist_ok=True)
    df.to_csv("../data/train_augmented.csv", index=False)
    print("Saved augmented dataset to data/train_augmented.csv")

generate_dataset_enhanced(2000)


Saved augmented dataset to data/train_augmented.csv
